## 1 · Import Required Libraries

In [1]:
import os
import json
import re
from typing import TypedDict, Optional, Literal
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END, START
from langgraph.constants import START, END

load_dotenv()

True

## 2 · Define State and Initialize LLM

Added `last_error` field to state so the query generator receives the validator's error message on retry instead of starting blind.

In [2]:
class AgentState(TypedDict):
    question: str
    sql_query: Optional[str]
    schema: Optional[str]
    results: Optional[list]
    answer: Optional[str]
    error_log: list
    retry_count: int
    is_relevant: bool
    last_error: Optional[str]   # NEW: validator error fed back to generator on retry

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

DB_USER = "admin"
DB_PASSWORD = "admin%40123"
DB_HOST = "100.90.162.48"
DB_PORT = "5432"
DB_NAME = "chicago_crime_new"
DB_URI = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

db = SQLDatabase.from_uri(DB_URI)
print(f"✓ Database connected: {db.dialect}")

✓ Database connected: postgresql


## 3 · Relevance Checker Node (unchanged from v2)

In [3]:
def relevance_checker(state: AgentState) -> AgentState:
    question = state["question"].lower().strip()

    query_action_verbs = {
        "how many", "how often", "what are the", "list", "show",
        "find", "get", "count", "total", "sum", "average", "top", "most", "least",
        "rank", "filter", "between", "after", "before", "during", "in the year", "percentage"
    }

    data_context_words = {
        "crime", "chicago", "crimes", "report", "reported", "database", "table",
        "records", "data", "district", "year", "date", "type", "arrest", "victim",
        "incidents", "cases", "statistics", "breakdown", "distribution"
    }

    irrelevant_patterns = [
        (r"what\s+is\s+\d+\s*[\+\-\*/]\s*\d+", "math expression"),
        (r"what is the meaning", "meaning of life"),
        (r"^what is\s+\w+\s*\?*\s*$", "generic what is"),
        (r"^define\s+", "define"),
        (r"^explain\s+", "explain"),
        (r"^tell me about\s+", "tell me about"),
        (r"who (are|is|was|were)", "who question"),
        (r"when (are|is|was|were)", "when question"),
        (r"why (are|is|was|were)", "why question"),
    ]

    invalid_data_questions = [
        (r"what\s+is\s+(chicago_crime|the chicago crime|the crimes|the database|the table)(?:\s|$|\?)", "non-specific data"),
    ]

    for pattern, reason in invalid_data_questions:
        if re.search(pattern, question, re.IGNORECASE):
            state["is_relevant"] = False
            state["error_log"].append(f"Relevance: {reason}")
            break
    else:
        for pattern, reason in irrelevant_patterns:
            if re.search(pattern, question, re.IGNORECASE):
                state["is_relevant"] = False
                state["error_log"].append(f"Relevance: {reason}")
                break
        else:
            has_action_verb = any(verb in question for verb in query_action_verbs)
            has_data_context = any(word in question for word in data_context_words)

            if has_action_verb and has_data_context:
                state["is_relevant"] = True
                state["error_log"].append("Relevance: Approved")
                return state

            state["is_relevant"] = False
            state["error_log"].append("Relevance: Rejected - missing action verb or data context")

    if not state["is_relevant"]:
        messages = [
            SystemMessage(content="You are a helpful assistant. The user's question is not related to a database or data analysis. Politely explain that their question is outside your scope and is not answerable using the database. Be concise but friendly."),
            HumanMessage(content=f"User question: {state['question']}")
        ]
        response = llm.invoke(messages)
        state["answer"] = response.content

    return state

## 4 · Schema Fetcher Node (unchanged from v2)

In [4]:
def schema_fetcher(state: AgentState) -> AgentState:
    try:
        schema = db.get_table_info()
        state["schema"] = schema
    except Exception as e:
        state["error_log"].append(f"Schema fetch error: {str(e)}")
        state["schema"] = ""
    return state

## 5 · Query Generator Node (v3 improvements)

**Changes from v2:**
1. System prompt explicitly documents the FK relationships between `crime_incidents` and lookup tables.
2. Instructs the LLM to ALWAYS join with `district_names` / `community_area_names` / `ward_names` when the user refers to a location by name — never hardcode numeric IDs.
3. Clarifies quoting rules: double quotes = identifiers only; single quotes = all string values.
4. On retry, the previous validator error is included in the prompt so the LLM knows what to fix.

In [5]:
SYSTEM_PROMPT = """You are a PostgreSQL expert. Write a precise SELECT query for the given question.

=== DATABASE RELATIONSHIPS ===
The crime_incidents table stores location as numeric codes:
  crime_incidents."DISTRICT"       (integer) → district_names."district_id"
  crime_incidents."COMMUNITY_AREA" (integer) → community_area_names."community_area_id"
  crime_incidents."WARD"           (integer) → ward_names."ward_id"

Lookup tables:
  district_names      : "district_id" (int PK), "district_name" (text)
  community_area_names: "community_area_id" (int PK), "community_area_name" (text)
  ward_names          : "ward_id" (int PK), "ward_name" (text)

=== CRITICAL RULES ===
1. QUOTING IDENTIFIERS: Enclose ALL column and table names in double quotes:
     "YEAR", "DISTRICT", "CRIME_TYPE", "crime_incidents", "district_names"

2. QUOTING STRING VALUES: ALWAYS use single quotes for string/text values:
     WHERE "CRIME_TYPE" = 'THEFT'      ← CORRECT
     WHERE "CRIME_TYPE" = "THEFT"      ← WRONG (double quotes = column name)
   Numeric values need no quotes: WHERE "YEAR" = 2020

3. RESOLVING LOCATION NAMES — most important rule:
   When the user mentions a district, community area, or ward BY NAME (e.g. "Central",
   "Lincoln Park", "Austin"), you MUST join with the lookup table and filter by name.
   NEVER hardcode numeric IDs or guess them.

   CORRECT example for "central district":
     SELECT COUNT(*) FROM "crime_incidents" ci
     JOIN "district_names" dn ON ci."DISTRICT" = dn."district_id"
     WHERE dn."district_name" ILIKE 'Central'
       AND ci."YEAR" = 2020;

   WRONG example:
     WHERE "DISTRICT" = '001'   ← hardcoded code, wrong

4. Use ILIKE for name matching so capitalisation doesn't matter.
5. Limit results to at most 3 rows unless the user asks for more.
6. Only select columns relevant to the question.
7. NEVER use INSERT / UPDATE / DELETE / DROP.
8. Return ONLY the SQL query, no markdown, no explanation.
   If the question cannot be answered with a SELECT, respond: INVALID_REQUEST
"""


def query_generator(state: AgentState) -> AgentState:
    schema = state.get("schema", "")
    question = state["question"]
    last_error = state.get("last_error")

    user_content = f"Schema:\n{schema}\n\nQuestion: {question}"

    # On retry, tell the LLM exactly what went wrong so it can fix the query
    if last_error:
        user_content += f"\n\nPrevious query failed validation with this error:\n{last_error}\nPlease fix the query."

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_content)
    ]
    response = llm.invoke(messages)
    sql_query = response.content.strip()

    if "INVALID_REQUEST" in sql_query:
        state["error_log"].append("Generator: Request cannot be fulfilled with SELECT query")
        state["sql_query"] = None
    else:
        state["sql_query"] = sql_query
        state["last_error"] = None  # clear after successful generation

    return state

## 6 · Query Validator Node

**Change from v2:** When validation fails, the error message is saved to `state["last_error"]` so `query_generator` can use it on the next attempt.

In [6]:
def query_validator(state: AgentState) -> AgentState:
    if not state.get("sql_query"):
        state["error_log"].append("Validator: No SQL query to validate")
        return state

    try:
        query = state["sql_query"]
        if re.search(r'\b(UPDATE|DELETE|DROP|INSERT|ALTER)\b', query, re.IGNORECASE):
            msg = f"Validator: DML statement detected in query: {query}"
            state["error_log"].append(msg)
            state["last_error"] = msg
            state["sql_query"] = None
            return state

        db.run(f"EXPLAIN {query}", fetch="all")
        state["error_log"].append("Validator: Query syntax is valid")
        state["last_error"] = None
    except Exception as e:
        error_msg = f"SQL syntax error: {str(e)}"
        state["error_log"].append(f"Validator: {error_msg}")
        state["last_error"] = error_msg   # feed back to generator on retry
        state["sql_query"] = None

    return state


def should_retry_query(state: AgentState) -> Literal["query_generator", "query_runner", "answer_synthesizer"]:
    if not state.get("sql_query"):
        state["retry_count"] += 1
        if state["retry_count"] >= 3:
            state["error_log"].append("Max retries reached (3). Ending query generation.")
            state["answer"] = "I was unable to generate a valid SQL query after 3 attempts. Please rephrase your question."
            return "answer_synthesizer"
        return "query_generator"
    return "query_runner"

## 7 · Query Runner Node (unchanged from v2)

In [7]:
def query_runner(state: AgentState) -> AgentState:
    try:
        result = db.run(state["sql_query"], fetch="all")
        state["results"] = result if isinstance(result, list) else [result]
        state["error_log"].append(f"Runner: Query executed successfully. Rows: {len(state['results'])}")
    except Exception as e:
        state["error_log"].append(f"Runner: Query execution error: {str(e)}")
        state["results"] = []
    return state

## 8 · Answer Synthesizer Node (unchanged from v2)

In [8]:
def answer_synthesizer(state: AgentState) -> AgentState:
    if state.get("answer"):
        return state

    if not state.get("results"):
        state["answer"] = "No results found for your query."
        return state

    results_str = json.dumps(state["results"][:10], indent=2)
    messages = [
        SystemMessage(content="You are a data analyst. Provide a clear, concise natural language summary of the SQL query results."),
        HumanMessage(content=f"Question: {state['question']}\n\nSQL Results:\n{results_str}\n\nProvide a summary.")
    ]
    response = llm.invoke(messages)
    state["answer"] = response.content
    return state

## 9 · Build StateGraph

In [9]:
def should_continue_after_relevance(state: AgentState) -> Literal["schema_fetcher", "__end__"]:
    return "schema_fetcher" if state.get("is_relevant") else END


graph = StateGraph(AgentState)

graph.add_node("relevance_checker", relevance_checker)
graph.add_node("schema_fetcher", schema_fetcher)
graph.add_node("query_generator", query_generator)
graph.add_node("query_validator", query_validator)
graph.add_node("query_runner", query_runner)
graph.add_node("answer_synthesizer", answer_synthesizer)

graph.add_edge(START, "relevance_checker")
graph.add_conditional_edges("relevance_checker", should_continue_after_relevance)
graph.add_edge("schema_fetcher", "query_generator")
graph.add_edge("query_generator", "query_validator")
graph.add_conditional_edges("query_validator", should_retry_query)
graph.add_edge("query_runner", "answer_synthesizer")
graph.add_edge("answer_synthesizer", END)

app = graph.compile()
print("✓ StateGraph compiled successfully")

✓ StateGraph compiled successfully


## 10 · Run Agent & Tests

In [10]:
def run_agent(question: str):
    initial_state: AgentState = {
        "question": question,
        "sql_query": None,
        "schema": None,
        "results": None,
        "answer": None,
        "error_log": [],
        "retry_count": 0,
        "is_relevant": False,
        "last_error": None,
    }

    result = app.invoke(initial_state)

    print(f"\n{'='*60}")
    print(f"Question: {result['question']}")
    print(f"SQL Query: {result.get('sql_query', 'N/A')}")
    print(f"Results: {result.get('results', [])}")
    print(f"Answer: {result.get('answer', 'N/A')}")
    print(f"Error Log: {result.get('error_log', [])}")
    print(f"{'='*60}\n")

    return result

print("Agent ready. Use run_agent('your question') to test.")

Agent ready. Use run_agent('your question') to test.


In [11]:
# Test 1: the failing case from v2 — district filter by name
result = run_agent("How many crimes has been reported in central district in 2020?")


Question: How many crimes has been reported in central district in 2020?
SQL Query: SELECT COUNT(*) AS "crime_count"
FROM "crime_incidents" ci
JOIN "district_names" dn ON ci."DISTRICT" = dn."district_id"
WHERE dn."district_name" ILIKE 'Central'
  AND ci."YEAR" = 2020;
Results: ['[(8428,)]']
Answer: In 2020, a total of **8,428 crimes** were reported in the Central district.
Error Log: ['Relevance: Approved', 'Validator: Query syntax is valid', 'Runner: Query executed successfully. Rows: 1']



In [12]:
# Test 2: community area by name (used to get string-quoting error in v2)
result = run_agent("How many thefts were reported in lincoln park in 2021?")


Question: How many thefts were reported in lincoln park in 2021?
SQL Query: SELECT COUNT(*) 
FROM "crime_incidents" ci
JOIN "community_area_names" ca ON ci."COMMUNITY_AREA" = ca."community_area_id"
WHERE ca."community_area_name" ILIKE 'Lincoln Park'
  AND ci."YEAR" = 2021
  AND ci."CRIME_TYPE" = 'THEFT';
Results: ['[(1160,)]']
Answer: In 2021, Lincoln Park had **1,160** reported theft incidents.
Error Log: ['Relevance: Approved', 'Validator: Query syntax is valid', 'Runner: Query executed successfully. Rows: 1']



In [13]:
# Test 3: multi-district join
result = run_agent("what are the second most common crime types in each district in 2020?")


Question: what are the second most common crime types in each district in 2020?
SQL Query: SELECT dn."district_name" AS "district",
       sub."CRIME_TYPE" AS "crime_type",
       sub.cnt AS "incident_count"
FROM (
    SELECT "DISTRICT",
           "CRIME_TYPE",
           COUNT(*) AS cnt,
           DENSE_RANK() OVER (PARTITION BY "DISTRICT" ORDER BY COUNT(*) DESC) AS rnk
    FROM "crime_incidents"
    WHERE "YEAR" = 2020
    GROUP BY "DISTRICT", "CRIME_TYPE"
) sub
JOIN "district_names" dn ON sub."DISTRICT" = dn."district_id"
WHERE sub.rnk = 2
LIMIT 3;
Results: ["[('Central', 'BATTERY', 1221), ('Wentworth', 'THEFT', 1863), ('Grand Crossing', 'THEFT', 1619)]"]
Answer: In 2020, the second‑most common crime in each district was:

- **Central** – Battery, with **1,221** incidents.  
- **Wentworth** – Theft, with **1,863** incidents.  
- **Grand Crossing** – Theft, with **1,619** incidents.
Error Log: ['Relevance: Approved', 'Validator: Query syntax is valid', 'Runner: Query executed succ

In [14]:
# Test 4: irrelevant question gate
result = run_agent("What is 2 + 2?")


Question: What is 2 + 2?
SQL Query: None
Results: None
Answer: I’m sorry, but that question isn’t something I can answer using the database tools I have available. If you have any queries related to data or analysis, I’d be happy to help!
Error Log: ['Relevance: math expression']

